# MS MARCO RARS-v4 Tri-state Action Feasibility

## tl;dr

This notebook runs the frozen RARS-v4 Phase-0 gate at implementation commit `bbdf8656881fd32d1961610d1c0b5c6d989fcc7a`. It first asks whether the source judgments actually preserve **explicit non-relevant** rows separately from **unjudged** candidates. Only then can it test protection, promotion, and penalty in the frozen post-PQ action space.

The expected result for a positive-only MS MARCO qrels cache is `STOP_NO_EXPLICIT_NEGATIVE_SEMANTICS`. That is a valid scientific stop, not a runtime failure. No Phase-0 result is training, QAT, an external confirmation, or a method-success claim.

## Frozen question and chronology

The proposed objective is only meaningful when all three states remain distinct: `+1` judged relevant, `-1` explicitly judged non-relevant, and `0` unjudged. Missing qrel rows are never converted to negatives.

The notebook reuses the completed v3 rank-32 progressive representation and matched-access comparator without refitting. It evaluates the already-observed 2,307-query v3 design role first. The already-observed 851-query diagnostic-audit labels are materialized only after a durable v4 design GO. The 803-query v3 future role remains identity-only and is never opened here.

A GO authorizes only a separately frozen FP32 development protocol with matched baselines and ablations.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

EXPERIMENT_PYTHON = sys.executable
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q',
    'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
], check=True)
NUMPY_TARGET = Path('/content/rars-v4-numpy126')
if NUMPY_TARGET.exists():
    shutil.rmtree(NUMPY_TARGET)
NUMPY_TARGET.mkdir(parents=True)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
    '--target', str(NUMPY_TARGET), 'numpy==1.26.4',
], check=True)
EXPERIMENT_ENV = os.environ.copy()
EXPERIMENT_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [
    str(NUMPY_TARGET), EXPERIMENT_ENV.get('PYTHONPATH', ''),
]))
installed_numpy_version = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__version__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert installed_numpy_version == '1.26.4', installed_numpy_version
installed_numpy_path = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__file__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert Path(installed_numpy_path).resolve().is_relative_to(NUMPY_TARGET.resolve())
print('Colab host-kernel NumPy (not used by experiments):',
      getattr(sys.modules.get('numpy'), '__version__', 'not-loaded'))
print('Fresh experiment-subprocess NumPy:', installed_numpy_version)

from google.colab import drive
drive.mount('/content/drive')

import hashlib, json

TRAINING_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
V3_IMPLEMENTATION_COMMIT = '05c2ae43b7d11783460822d10c590240dab1a399'
V4_IMPLEMENTATION_COMMIT = 'bbdf8656881fd32d1961610d1c0b5c6d989fcc7a'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
TRAIN_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
V3_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v3')
V4_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v4')

WORK = Path('/content') / f'rars-v4-{V4_IMPLEMENTATION_COMMIT[:12]}'
PARENT_WORK = Path('/content') / f'rars-v2.2-{TRAINING_COMMIT[:12]}'
for local_work in (WORK, PARENT_WORK):
    if local_work.exists():
        shutil.rmtree(local_work)
    local_work.mkdir(parents=True)
PARENT_BUNDLES = PARENT_WORK / 'bundles'
PARENT_CANDIDATE_CACHE = PARENT_WORK / 'candidate-cache'
V3_BUNDLES = WORK / 'v3-bundles'
V4_LABELS = WORK / 'tristate-labels'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
V3_OUTPUT = DRIVE / 'rars-v3-oracle-first' / V3_IMPLEMENTATION_COMMIT[:12]
OUTPUT = DRIVE / 'rars-v4-tristate-action-feasibility' / V4_IMPLEMENTATION_COMMIT[:12]

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_record(path, record):
    path = Path(path)
    assert path.is_file(), path
    assert path.stat().st_size == int(record['bytes']), path
    assert sha256_file(path) == record['sha256'], path

EXPERIMENT_PROBE = r'''
import json, sys
import faiss
import numpy as np
print(json.dumps({
    'python_version': '.'.join(map(str, sys.version_info[:3])),
    'numpy_version': np.__version__,
    'numpy_module_path': np.__file__,
    'faiss_version': str(getattr(faiss, '__version__', 'UNKNOWN')),
}, allow_nan=False))
'''

In [ ]:
def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    head = subprocess.check_output(
        ['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True
    ).strip()
    dirty = subprocess.check_output(
        ['git', '-C', str(destination), 'status', '--porcelain'], text=True
    ).strip()
    assert head == commit, (head, commit)
    assert not dirty, dirty

clone_exact(TRAIN_REPO, TRAINING_COMMIT)
clone_exact(V3_REPO, V3_IMPLEMENTATION_COMMIT)
clone_exact(V4_REPO, V4_IMPLEMENTATION_COMMIT)

V3_PROTOCOL_PATH = V3_REPO / 'protocols/rars_v3_oracle_first_feasibility_v1.json'
PROTOCOL_PATH = V4_REPO / 'protocols/rars_v4_tristate_action_feasibility_v1.json'
protocol = json.loads(PROTOCOL_PATH.read_text())
assert protocol['status'] == 'FROZEN_BEFORE_FIRST_TRISTATE_LABEL_AUDIT'
assert protocol['method_rationale_is_outcome_informed'] is True
assert protocol['method_revision_allowed'] is False
assert protocol['outcome_informed_revision_allowed'] is False
assert protocol['parent_lineage']['v3_implementation_commit'] == V3_IMPLEMENTATION_COMMIT
assert protocol['data_policy']['roles']['future_fp32_holdout']['candidate_arrays_allowed_in_phase0'] is False

subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
], cwd=TRAIN_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v3_oracle_core.py',
    'tests/test_build_msmarco_rars_v3_oracle_bundles.py',
], cwd=V3_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v4_tristate_action_core.py',
    'tests/test_materialize_rars_v4_tristate_labels.py',
    'tests/test_rars_v4_tristate_protocol_contract.py',
    'tests/test_evaluate_rars_v4_tristate_action_feasibility.py',
], cwd=V4_REPO, check=True, env=EXPERIMENT_ENV)
print('Exact v4 implementation commit:', V4_IMPLEMENTATION_COMMIT)
print('Frozen v4 protocol SHA-256:', sha256_file(PROTOCOL_PATH))

In [ ]:
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
    V3_OUTPUT / 'oracle_complete.json',
    V3_OUTPUT / 'oracle_summary.json',
    V3_OUTPUT / 'design_freeze.json',
    V3_OUTPUT / 'progressive_svd_rank32.float32.npy',
    V3_OUTPUT / 'progressive_svd_rank32_scales.float32.npy',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
assert shutil.disk_usage('/content').free >= 4_000_000_000, 'Need 4 GB local disk'
current_environment = json.loads(subprocess.check_output(
    [EXPERIMENT_PYTHON, '-c', EXPERIMENT_PROBE],
    text=True, env=EXPERIMENT_ENV,
))
contract = protocol['execution_environment_contract']
assert current_environment['python_version'] == contract['python_version']
assert current_environment['numpy_version'] == contract['numpy_version']
assert Path(current_environment['numpy_module_path']).resolve().is_relative_to(
    NUMPY_TARGET.resolve()
)
assert current_environment['faiss_version'] != 'UNKNOWN'
v3_summary = json.loads((V3_OUTPUT / 'oracle_summary.json').read_text())
assert v3_summary['formal_decision'] == protocol['parent_lineage']['v3_observed_formal_decision']
print(json.dumps(current_environment, indent=2))

## Data and labels

The inherited v2.2 bundle is reproduced exactly so the v3 qrels-free candidates can be recreated. RARS-v4 does **not** read the old binary candidate labels. Its separate materializer reads the source JSON and preserves graded mapping rows as P/N/U.

The v3 builder still writes only identity files for `future_method_holdout`; no Phase-0 cell below opens that role.

In [ ]:
builder = [
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(PARENT_CANDIDATE_CACHE),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(PARENT_BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(PARENT_BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(TRAIN_REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', TRAINING_COMMIT,
], check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
parent = json.loads(V3_PROTOCOL_PATH.read_text())['parent_lineage']
parent_hashes = {
    'parent_inner_train_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/v2_2_manifest.json'),
    'parent_inner_train_source_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/manifest.json'),
    'parent_inner_train_query_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/query_manifest.json'),
    'closed_inner_validation_query_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_validation/query_manifest.json'),
    'parent_v2_2_split_audit_sha256': sha256_file(PARENT_BUNDLES / 'v2_2_split_audit.json'),
}
for key, actual in parent_hashes.items():
    assert actual == parent[key], (key, actual, parent[key])
print('Exact v2.2 parent rematerialized.')

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/build_msmarco_rars_v3_oracle_bundles.py'),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--output-root', str(V3_BUNDLES),
    '--protocol', str(V3_PROTOCOL_PATH),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--n-docs', '1000000',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
candidate_summary = json.loads(
    (V3_BUNDLES / 'v3_oracle_bundle_freeze_summary.json').read_text()
)
assert candidate_summary['status'] == 'V3_QRELS_FREE_CANDIDATE_BUNDLES_FROZEN'
assert candidate_summary['parent_label_payload_bytes_read'] is False
assert candidate_summary['qrels_opened_or_parsed'] is False
assert candidate_summary['future_method_holdout']['candidate_arrays_created'] is False
future_files = {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()}
assert future_files == {'query_manifest.json', 'v3_identity_manifest.json'}
print('V3 design/audit candidates recreated; future role remains identity-only.')

In [ ]:
DESIGN_LABEL_DIR = V4_LABELS / 'v4_design_observed'
AUDIT_LABEL_DIR = V4_LABELS / 'v4_diagnostic_audit'
subprocess.run([
    EXPERIMENT_PYTHON, str(V4_REPO / 'scripts/materialize_rars_v4_tristate_labels.py'),
    '--candidate-bundle', str(V3_BUNDLES / 'oracle_design'),
    '--judgments', str(CACHE / 'qrels_subset.json'),
    '--output-dir', str(DESIGN_LABEL_DIR),
    '--role', 'v4_design_observed',
    '--source-commit', V4_IMPLEMENTATION_COMMIT,
    '--protocol', str(PROTOCOL_PATH),
], check=True, cwd=V4_REPO, env=EXPERIMENT_ENV)
design_label_manifest = DESIGN_LABEL_DIR / 'v4_tristate_labels_manifest.json'
design_labels = json.loads(design_label_manifest.read_text())
assert design_labels['binary_candidate_relevance_read'] is False
assert design_labels['missing_rows_interpreted_as_explicit_negative'] is False
assert not AUDIT_LABEL_DIR.exists()
print(json.dumps(design_labels['source_schema'], indent=2))

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON, str(V4_REPO / 'scripts/evaluate_rars_v4_tristate_action_feasibility.py'),
    '--phase', 'design',
    '--candidate-bundle', str(V3_BUNDLES / 'oracle_design'),
    '--label-manifest', str(design_label_manifest),
    '--v3-output-dir', str(V3_OUTPUT),
    '--protocol', str(PROTOCOL_PATH),
    '--output-dir', str(OUTPUT),
    '--source-commit', V4_IMPLEMENTATION_COMMIT,
    '--reuse-complete',
], check=True, cwd=V4_REPO, env=EXPERIMENT_ENV)
design_summary = json.loads((OUTPUT / 'design_summary.json').read_text())
design_freeze = json.loads((OUTPUT / 'design_freeze.json').read_text())
assert design_summary['future_method_holdout_accessed'] is False
assert design_freeze['future_method_holdout_accessed'] is False
RUN_DIAGNOSTIC_AUDIT = (
    design_summary['formal_decision'] == 'DESIGN_GO_TO_DIAGNOSTIC_AUDIT'
)
print(json.dumps({
    'formal_decision': design_summary['formal_decision'],
    'explicit_negative_semantics_preserved': design_summary['explicit_negative_semantics_preserved'],
    'source_schema': design_summary['label_source_schema'],
    'coverage': design_summary['label_support']['coverage'],
    'run_diagnostic_audit': RUN_DIAGNOSTIC_AUDIT,
}, indent=2))

## Conditional diagnostic audit

The next cell is intentionally conditional. If the design source is positive-only or any label-support gate fails, it prints a formal stop and does **not** materialize the 851-query diagnostic-audit labels. A design GO is the only condition that permits that second label release.

In [ ]:
if RUN_DIAGNOSTIC_AUDIT:
    subprocess.run([
        EXPERIMENT_PYTHON, str(V4_REPO / 'scripts/materialize_rars_v4_tristate_labels.py'),
        '--candidate-bundle', str(V3_BUNDLES / 'oracle_audit'),
        '--judgments', str(CACHE / 'qrels_subset.json'),
        '--output-dir', str(AUDIT_LABEL_DIR),
        '--role', 'v4_diagnostic_audit',
        '--source-commit', V4_IMPLEMENTATION_COMMIT,
        '--protocol', str(PROTOCOL_PATH),
        '--design-freeze', str(OUTPUT / 'design_freeze.json'),
    ], check=True, cwd=V4_REPO, env=EXPERIMENT_ENV)
    audit_label_manifest = AUDIT_LABEL_DIR / 'v4_tristate_labels_manifest.json'
    subprocess.run([
        EXPERIMENT_PYTHON, str(V4_REPO / 'scripts/evaluate_rars_v4_tristate_action_feasibility.py'),
        '--phase', 'audit',
        '--candidate-bundle', str(V3_BUNDLES / 'oracle_audit'),
        '--label-manifest', str(audit_label_manifest),
        '--v3-output-dir', str(V3_OUTPUT),
        '--protocol', str(PROTOCOL_PATH),
        '--output-dir', str(OUTPUT),
        '--source-commit', V4_IMPLEMENTATION_COMMIT,
        '--reuse-complete',
    ], check=True, cwd=V4_REPO, env=EXPERIMENT_ENV)
    print('One-shot diagnostic audit completed or exact complete run verified.')
else:
    assert not AUDIT_LABEL_DIR.exists()
    print('Diagnostic audit correctly skipped:', design_summary['formal_decision'])

In [ ]:
if RUN_DIAGNOSTIC_AUDIT:
    final_summary = json.loads((OUTPUT / 'phase0_summary.json').read_text())
    marker = json.loads((OUTPUT / 'phase0_complete.json').read_text())
else:
    final_summary = design_summary
    marker = design_freeze
assert final_summary['future_method_holdout_accessed'] is False
assert final_summary['training_allowed'] is False
assert final_summary['qat_allowed'] is False
assert final_summary['external_evaluation_allowed'] is False
assert final_summary['go_is_method_success'] is False
assert marker['run_fingerprint'] == final_summary['run_fingerprint']
root = OUTPUT.resolve()
for relative_name, record in marker['registered_outputs'].items():
    relative = Path(relative_name)
    assert not relative.is_absolute() and '..' not in relative.parts
    path = (OUTPUT / relative).resolve()
    assert root in path.parents
    verify_record(path, record)
report = {
    'formal_decision': final_summary['formal_decision'],
    'evidence_status': final_summary['evidence_status'],
    'explicit_negative_semantics_preserved': final_summary['explicit_negative_semantics_preserved'],
    'source_schema': final_summary['label_source_schema'],
    'label_support_coverage': final_summary['label_support']['coverage'],
    'label_swap_ceiling': final_summary['label_support']['label_swap_ceiling'],
    'pre_action_gate': final_summary['pre_action_gate'],
    'progressive_action_space': final_summary['progressive_action_space'],
    'future_method_holdout_accessed': False,
    'output_dir': str(OUTPUT),
}
print(json.dumps(report, indent=2, allow_nan=False))

## Takeaways

Interpret the formal decision literally. In particular:

- `STOP_NO_EXPLICIT_NEGATIVE_SEMANTICS` means the current cache cannot test the proposed negative penalty. Do not replace unjudged candidates with fake explicit negatives.
- `STOP_NO_NOVEL_ACTION_SUPPORT` means the new unary term would add too little independent supervision beyond v2.2.
- `STOP_NO_COMPRESSION_CONSISTENT_HEADROOM` means labels support swaps but the frozen post-PQ actions cannot realize them broadly enough.
- `GO_FREEZE_FP32_DEVELOPMENT_PROTOCOL` only permits writing a separate matched-baseline FP32 protocol. It does not permit QAT, external testing, or a success claim.

Return the printed report and, if present, `design_summary.json`, `design_freeze.json`, `phase0_summary.json`, and `phase0_complete.json` for review.